In [6]:
import os, glob, importlib.util

src_path = "../src"
for file in glob.glob(os.path.join(src_path, "*.py")):
    module_name = os.path.splitext(os.path.basename(file))[0]
    spec = importlib.util.spec_from_file_location(module_name, file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    globals()[module_name] = module


In [1]:
import pandas as pd
sentiment_df  = pd.read_csv("../data/processed/macro_sentiment.csv")
df  = pd.read_csv("../data/processed/features_panel_data.csv")

In [3]:
# ==========================================
# PRE-MERGE STATE
# ==========================================
rows_before = len(df)

# Drop existing sentiment columns from df to avoid duplicate conflicts
df = df.drop(columns=['Macro_Sentiment', 'Macro_Sentiment_3d', 'Macro_News_Volume'], errors='ignore')

# Keep only Date + sentiment columns in sentiment_df
sentiment_df = sentiment_df[['Date', 'Macro_Sentiment', 'Macro_Sentiment_3d', 'Macro_News_Volume']]

# Execute the merge
df = pd.merge(df, sentiment_df, on='Date', how='left')

# ==========================================
# MERGE SANITY CHECK PROTOCOL
# ==========================================
print("\n--- MERGE SANITY CHECK ---")

# Check 1: Row Explosion/Loss
rows_after = len(df)
if rows_before != rows_after:
    print(f"🚨 WARNING: Row count changed! Before: {rows_before}, After: {rows_after}")
    print("Check your sentiment dataset for duplicate dates.")
else:
    print("✅ PASS: Row count remained stable.")

# Check 2: Null Value Mapping
missing_sentiment = df['Macro_Sentiment'].isna().sum()
match_rate = 100 - ((missing_sentiment / rows_after) * 100)
print(f"📊 Sentiment Match Rate: {match_rate:.2f}% ({missing_sentiment} days missing)")

if match_rate < 80:
    print("🚨 WARNING: Low match rate. Check if Date formatting aligns (e.g., Timezones).")

# Fill the missing days safely
df['Macro_Sentiment'] = df['Macro_Sentiment'].fillna(0.0)
df['Macro_Sentiment_3d'] = df['Macro_Sentiment_3d'].fillna(0.0)
df['Macro_News_Volume'] = df['Macro_News_Volume'].fillna(0)

# Check 3: Enforce Sorting
df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
print("✅ PASS: Re-enforced Ticker -> Date sorting.")

# Check 4: Visual Spot Check
print("\n--- SPOT CHECK (First 3 days of AAPL) ---")
print(df[df['Ticker'] == 'AAPL'][['Date', 'Close', 'Macro_Sentiment']].head(3))
print("--------------------------\n")



--- MERGE SANITY CHECK ---
✅ PASS: Row count remained stable.
📊 Sentiment Match Rate: 82.22% (2200 days missing)
✅ PASS: Re-enforced Ticker -> Date sorting.

--- SPOT CHECK (First 3 days of AAPL) ---
         Date      Close  Macro_Sentiment
0  2020-01-31  74.539894         0.788400
1  2020-02-03  74.335175         0.334743
2  2020-02-04  76.789253         0.118600
--------------------------

